In [ ]:
import random
import re
from datasets import load_dataset, get_dataset_config_names
from typing import List

# ===============================================================================
# 🌟 데이터셋 개요 (Annotation Block)
# 📘 제목: 한국어 위키피디아 데이터셋 (Korean Wikipedia Dataset)
# 💡 의미: GPT-2와 같은 대규모 언어 모델(LLM)을 학습시키기 위해 수집된
#       방대한 한국어 위키피디아 기사 텍스트 데이터입니다.
# ✨ 설명: 이 데이터셋은 언어 모델이 한국어 문맥, 문장 구조, 그리고 지식을
#       자연스럽게 학습할 수 있도록 도와주는 핵심적인 "교재" 역할을 합니다.
#       우리는 이 데이터를 활용하여 텍스트 패턴을 분석하고, AI가 다음 단어를
#       어떻게 예측하는지 '미리 체험'해 볼 거예요!
# ===============================================================================


# --- 설정 변수 ---
DATASET_NAME = "eaglewatch/Korean_Wikipedia_Dataset_for_GPT2_August_2022"
SAMPLE_COUNT = 100  # 실습을 위해 상위 100개 샘플만 사용합니다. (효율성!)

# -------------------------------------------------------------------------------
# 🚀 1. 데이터 로딩 및 준비 (친절한 에러 처리 필수!)
# -------------------------------------------------------------------------------

print("✨ [Step 1] 데이터를 로드할 준비를 합니다. 스트리밍 모드를 먼저 시도해 볼게요!")

dataset = None
dataset_type = "Unknown"

try:
    # 💡 시도 1: 스트리밍 모드 (가장 빠르고 메모리 효율적!)
    print("   -> 스트리밍 모드(streaming=True)로 로딩을 시도합니다...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    dataset_type = "Streaming Dataset (Iterable)"
    print("✅ 성공! 스트리밍 모드로 데이터를 불러왔습니다. 메모리 걱정 끝!")

except Exception as e:
    # 🆘 시도 2: 스트리밍 실패 시 (혹은 특정 환경에서 에러가 날 경우)
    print(f"   [경고] 스트리밍 로딩에 문제가 발생했습니다: {e}")
    print("   -> 메모리 부담이 적은 작은 분할(split='train[:500]')을 일반 모드로 로드해 볼게요.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train[:500]', streaming=False)
        dataset_type = "Standard Dataset"
        print("✅ 성공! 일반 모드로 데이터를 불러왔습니다. 이제 마음껏 분석할 수 있어요!")
    except Exception as e_fallback:
        print(f"❌ 치명적인 에러 발생: 데이터셋 로딩 실패! {e_fallback}")
        exit()


print("-" * 50)
print(f"📊 로딩 성공! 사용된 데이터 타입: {dataset_type}")


# -------------------------------------------------------------------------------
# ♻️ 2. 샘플링 전략 수립 (핵심 원칙 준수!)
# -------------------------------------------------------------------------------

# 💡 목표: 데이터셋 전체를 사용하지 않고, 상위 K개만 안전하게 추출합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)입니다.
    print(f"✨ 상위 {SAMPLE_COUNT}개 샘플을 순차적으로 추출합니다...")
    # 🚀 스트리밍 데이터를 위한 안전한 방법: take(K)를 이용한 이터레이터 생성
    sample_iterator = iter(dataset.take(SAMPLE_COUNT))
    # list()로 변환하여 반복문에서 사용하기 쉽게 만듭니다.
    sampled_dataset = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)인 경우
    print(f"✨ 상위 {SAMPLE_COUNT}개 샘플을 추출합니다...")
    sampled_dataset = list(dataset.select(range(min(SAMPLE_COUNT, 1000)))) # 안전하게 1000개 범위에서 가져오기


print(f"✅ 준비 완료! 총 {len(sampled_dataset)}개의 샘플을 분석에 사용할 거예요.")
print("-" * 50)


# -------------------------------------------------------------------------------
# 🎨 3. 초급 AI 실습 1: 텍스트 통계 분석 및 EDA (탐험가 모드)
# -------------------------------------------------------------------------------

def analyze_text_stats(data_list: List[dict]):
    """주어진 샘플 데이터 목록을 분석하여 통계 정보를 출력합니다."""
    print("\n======== 🔬 실습 1: 데이터 텍스트 통계 분석 (EDA) ========")
    
    # 🌐 모든 텍스트를 모아서 한 번에 처리하는 것이 효율적입니다.
    all_text = ""
    for sample in data_list:
        # 데이터 필드 이름이 'text'인 것을 가정합니다.
        if 'text' in sample:
            all_text += sample['text'] + " "
        else:
            print("⚠️ 'text' 필드를 찾을 수 없습니다. 데이터 구조를 확인해 주세요.")
            return

    total_characters = len(all_text)
    total_words = len(all_text.split())
    
    # 📝 글자 수와 단어 수라는 정량적 지표를 계산합니다.
    print(f"📊 📝 총 분석 텍스트 길이: {total_characters:,} 글자")
    print(f"🗣️ 📚 총 단어 수: {total_words:,} 단어")

    # ✨ 가장 많이 사용된 글자 5개 찾기 (단순 빈도 분석)
    char_counts = {}
    for char in re.findall(r'[가-힣a-zA-Z0-9]', all_text):
        char_counts[char] = char_counts.get(char, 0) + 1
        
    # sorted()를 사용하여 빈도가 높은 순서로 정렬
    most_common_chars = sorted(char_counts.items(), key=lambda item: item[1], reverse=True)
    
    print("\n⭐ 가장 자주 등장하는 글자 TOP 5:")
    for i, (char, count) in enumerate(most_common_chars[:5]):
        print(f"  - '{char}': {count:,}회")


# -------------------------------------------------------------------------------
# 💡 4. 초급 AI 실습 2: 프롬프트 기반 패턴 예측 시뮬레이션 (작가 모드)
# -------------------------------------------------------------------------------

def simulate_completion(data_list: List[dict]):
    """
    위키피디아 텍스트에서 무작위로 문장을 뽑아, 다음 단어를 예측하는 과정을 시뮬레이션합니다.
    (LLM의 'Infilling' 또는 'Completion' 작업의 맛보기입니다!)
    """
    print("\n======== 🔮 실습 2: 텍스트 완성도 예측 시뮬레이션 (Completion) ========")
    
    if not data_list:
        print("분석할 샘플이 없습니다.")
        return

    # 💡 목표: 임의의 문장 1개를 선택하여, AI가 어떤 단어로 이어갈지 추측해봅니다.
    sample_text = random.choice(data_list)['text']
    
    print("\n👉 [AI 추론 시작]: 다음 문장을 완성해 보세요!")
    print(f"   << 원문 문장: '{sample_text[:100]}...' >>")
    
    # ✂️ 임의의 분기점을 찾습니다. (예: 마침표 '.' 다음)
    try:
        # 문장 중간에 '.' (점)을 강제로 넣어 끊어봅니다.
        fragment_index = sample_text.rfind('.')
        if fragment_index == -1 or fragment_index < 10:
             # 만약 점이 없거나 너무 앞에 있다면, 아예 잘라냅니다.
             print("✨ 점이 없어 문장 앞부분만 활용합니다.")
             prompt = sample_text[:min(len(sample_text), 150)]
             completion_context = " (이어지는 문맥)"
        else:
             prompt = sample_text[:fragment_index+1]
             completion_context = sample_text[fragment_index+1:fragment_index+30].replace(" ", "") + "..."

        print("-" * 50)
        print(f"    ✏️ 프롬프트(Prompt): \"{prompt}\"")
        print(f"    ✅ 예상 컨텍스트: {completion_context}")
        print("\n⭐ Tutter's Tip: 위키피디아는 전 세계 지식이 담겨있기에,")
        print("  AI는 이 패턴을 보고 자연스러운 '요약'이나 '다음 지식'을 완성합니다!")
    except Exception as e:
        print(f"패턴 추출 중 오류가 발생했습니다: {e}")


# -------------------------------------------------------------------------------
# 🧪 5. 초급 AI 실습 3: 텍스트 전처리 및 데이터 압축 (가공 전문가 모드)
# -------------------------------------------------------------------------------

def preprocess_text(text: str) -> str:
    """
    텍스트를 정제하는 함수입니다. (불필요한 기호, 띄어쓰기 오류 등을 수정)
    """
    # 1. HTML 태그 제거 (위키피디아 특유의 <br> 같은 태그 정리)
    cleaned_text = re.sub(r'<\/?\w+[^>]*>', '', text)
    # 2. 특수문자 및 숫자 정리 (자연어 모델 학습을 위해 문자에 집중)
    cleaned_text = re.sub(r'[^가-힣\s.,!?]', '', cleaned_text)
    # 3. 과도한 공백 제거
    cleaned_text = ' '.join(cleaned_text.split())
    return cleaned_text.strip()

def run_preprocessing_example(data_list: List[dict]):
    """전처리 함수를 사용하여 샘플 텍스트 2개를 비교합니다."""
    print("\n======== ⚙️ 실습 3: 텍스트 전처리 및 정제 (Pre-processing) ========")
    
    if not data_list:
        print("분석할 샘플이 없어 건너뜁니다.")
        return

    # 📚 샘플 1 (Original)
    original_text = data_list[0]['text']
    processed_text = preprocess_text(original_text)
    
    print("\n✨ 샘플 1 비교:")
    print(f"  [원본 텍스트 (절단)]: {original_text[:200]}...")
    print(f"  [전처리된 텍스트]: {processed_text[:200]}...")
    print("💡 주석: `re.sub` 같은 정규표현식(Regular Expression)은 AI가 이해하는")
    print("  '깨끗한' 언어 형태로 데이터를 다듬는 필수 도구입니다.")

    # 📚 샘플 2 (Another sample)
    original_text_2 = data_list[1]['text']
    processed_text_2 = preprocess_text(original_text_2)

    print("\n✨ 샘플 2 비교:")
    print(f"  [원본 텍스트 (절단)]: {original_text_2[:200]}...")
    print(f"  [전처리된 텍스트]: {processed_text_2[:200]}...")
    
    print("\n🎉 수고하셨어요! 🎉")
    print("이 세 가지 실습(통계 분석, 패턴 완성, 전처리)을 통해")
    print("텍스트 데이터로 어떤 '마법'을 부릴 수 있는지 맛보았답니다.")

# -------------------------------------------------------------------------------
# 🧪 6. 메인 실행 로직 (실습 함수 호출)
# -------------------------------------------------------------------------------
if __name__ == "__main__":
    # 1. 통계 분석 실행
    analyze_text_stats(sampled_dataset)
    
    # 2. 패턴 예측 시뮬레이션 실행
    simulate_completion(sampled_dataset)
    
    # 3. 전처리 예제 실행
    run_preprocessing_example(sampled_dataset)